# DeepTruth — Stage 4: Video Temporal Model Training

**Run on Google Colab Pro (A100 GPU recommended)**

### What this notebook does
- Loads the best image model checkpoint from notebook 03
- **Freezes** all 5 streams and the cross-attention fusion
- **Trains only the Temporal Transformer** on video frame sequences
- Then unfreezes everything for full end-to-end fine-tuning on video

### Prerequisites
- Complete notebooks 02 and 03 first
- Upload `deeptruth_data.zip` to Drive (must include video sequences)
- Best image checkpoint at `Drive/DeepTruth/checkpoints/deeptruth_image_model_final.pth`

In [ ]:
# Cell 1 — Setup
from google.colab import drive
drive.mount('/content/drive')

!pip install -q timm open-clip-torch albumentations einops

import os, json, math, random, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from torchmetrics import AUROC, Accuracy

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DRIVE_ROOT = '/content/drive/MyDrive/DeepTruth'
DATA_ZIP   = f'{DRIVE_ROOT}/deeptruth_data.zip'
DATA_DIR   = '/content/deeptruth_data'
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 2 — Unpack data
if not os.path.exists(DATA_DIR):
    print('Unpacking data...')
    !unzip -q {DATA_ZIP} -d {DATA_DIR}
    print('Done.')
else:
    print('Data already unpacked.')

# Discover video sequences
VIDEO_SEQ_DIR = os.path.join(DATA_DIR, 'video_sequences')
if not os.path.exists(VIDEO_SEQ_DIR):
    # Try alternative paths
    for candidate in [
        os.path.join(DATA_DIR, 'sequences'),
        os.path.join(DATA_DIR, 'raw', 'video'),
        os.path.join(DATA_DIR, 'video'),
    ]:
        if os.path.exists(candidate):
            VIDEO_SEQ_DIR = candidate
            break

print(f'Video sequence dir: {VIDEO_SEQ_DIR}')
if os.path.exists(VIDEO_SEQ_DIR):
    subdirs = os.listdir(VIDEO_SEQ_DIR)
    print(f'Subdirs: {subdirs[:10]}')

In [ ]:
# Cell 3 — Video dataset class
# Each sample = a sequence of N_FRAMES face crops from one video

N_FRAMES = 8        # frames per clip (8 is sufficient for DFD face-swap detection)
IMG_SIZE = 224

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

FAKE_TYPE_MAP = {
    'real': 0,
    'Deepfakes': 1,
    'Face2Face': 2,
    'FaceSwap': 3,
    'NeuralTextures': 4,
    'DeepFakeDetection': 5,
    'FaceShifter': 6,
    'unknown_fake': 9,
}

def get_frame_transform(train=True):
    if train:
        return A.Compose([
            A.RandomResizedCrop(IMG_SIZE, IMG_SIZE, scale=(0.85, 1.0)),
            A.HorizontalFlip(p=0.5),
            A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, p=0.4),
            A.GaussNoise(var_limit=(5, 30), p=0.3),
            A.ImageCompression(quality_lower=70, quality_upper=100, p=0.3),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])


class VideoSequenceDataset(Dataset):
    """
    Expects data organised as:
      video_sequences/
        real/
          <video_id>/
            frame_000.jpg  ...
        fake/
          <manipulation_type>/
            <video_id>/
              frame_000.jpg  ...

    Falls back to flat DFD-style structure.
    """
    def __init__(self, seq_dir, split='train', n_frames=N_FRAMES,
                 transform=None, temporal_jitter=True):
        self.n_frames = n_frames
        self.transform = transform
        self.temporal_jitter = temporal_jitter
        self.sequences = []
        self._discover(seq_dir, split)

    def _discover(self, seq_dir, split):
        for label_name, label in [('real', 0), ('fake', 1)]:
            label_dir = os.path.join(seq_dir, label_name)
            if not os.path.isdir(label_dir):
                for sub in os.listdir(seq_dir):
                    subpath = os.path.join(seq_dir, sub)
                    if not os.path.isdir(subpath):
                        continue
                    is_fake = 'manipulated' in sub.lower() or 'fake' in sub.lower()
                    is_real = 'original' in sub.lower() or 'real' in sub.lower()
                    if not (is_fake or is_real):
                        continue
                    lbl = 1 if is_fake else 0
                    ftype = 'unknown_fake' if is_fake else 'real'
                    for vname in os.listdir(subpath):
                        vpath = os.path.join(subpath, vname)
                        if os.path.isdir(vpath):
                            if len(self._get_frames(vpath)) >= 4:
                                self.sequences.append((vpath, lbl, ftype))
                return

            for entry in os.listdir(label_dir):
                entry_path = os.path.join(label_dir, entry)
                if os.path.isdir(entry_path):
                    sub_items = os.listdir(entry_path)
                    if any(f.endswith(('.jpg', '.png')) for f in sub_items):
                        ftype = entry if label == 1 else 'real'
                        if len(self._get_frames(entry_path)) >= 4:
                            self.sequences.append((entry_path, label, ftype))
                    else:
                        fake_type = entry if label == 1 else 'real'
                        for vname in sub_items:
                            vpath = os.path.join(entry_path, vname)
                            if os.path.isdir(vpath):
                                if len(self._get_frames(vpath)) >= 4:
                                    self.sequences.append((vpath, label, fake_type))

        random.seed(42)
        random.shuffle(self.sequences)
        n = len(self.sequences)
        if split == 'train':
            self.sequences = self.sequences[:int(0.8 * n)]
        elif split == 'val':
            self.sequences = self.sequences[int(0.8 * n):int(0.9 * n)]
        else:
            self.sequences = self.sequences[int(0.9 * n):]
        print(f'{split}: {len(self.sequences)} video sequences')

    def _get_frames(self, folder):
        return sorted(
            [os.path.join(folder, f) for f in os.listdir(folder)
             if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        )

    def _sample_frames(self, frame_paths):
        n = len(frame_paths)
        if n >= self.n_frames:
            if self.temporal_jitter:
                indices = np.linspace(0, n - 1, self.n_frames, dtype=float)
                jitter  = np.random.uniform(-0.4, 0.4, size=len(indices))
                indices = np.clip(indices + jitter, 0, n - 1).astype(int)
            else:
                indices = np.linspace(0, n - 1, self.n_frames, dtype=int)
        else:
            indices = list(range(n))
            while len(indices) < self.n_frames:
                indices += indices
            indices = indices[:self.n_frames]
        return [frame_paths[i] for i in indices]

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        folder, label, fake_type = self.sequences[idx]
        all_frames = self._get_frames(folder)
        selected   = self._sample_frames(all_frames)

        frames_list = []
        for fp in selected:
            img = Image.open(fp).convert('RGB')
            img_np = np.array(img)
            if self.transform:
                frame = self.transform(image=img_np)['image'].float()
            else:
                import torchvision.transforms as T
                frame = T.ToTensor()(img)
            frames_list.append(frame)

        video_tensor = torch.stack(frames_list, dim=0)  # (T, C, H, W)
        type_id = FAKE_TYPE_MAP.get(fake_type, FAKE_TYPE_MAP['unknown_fake'])
        return video_tensor, torch.tensor(label, dtype=torch.long), torch.tensor(type_id, dtype=torch.long)

print('VideoSequenceDataset defined.')

In [ ]:
# Cell 4 — Build video DataLoaders
BATCH_SIZE  = 8   # 8 frames per sample is lighter — can use larger batch
NUM_WORKERS = 2

train_vds = VideoSequenceDataset(VIDEO_SEQ_DIR, split='train',
                                  transform=get_frame_transform(train=True),
                                  temporal_jitter=True)
val_vds   = VideoSequenceDataset(VIDEO_SEQ_DIR, split='val',
                                  transform=get_frame_transform(train=False),
                                  temporal_jitter=False)
test_vds  = VideoSequenceDataset(VIDEO_SEQ_DIR, split='test',
                                  transform=get_frame_transform(train=False),
                                  temporal_jitter=False)

train_vloader = DataLoader(train_vds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_vloader   = DataLoader(val_vds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
test_vloader  = DataLoader(test_vds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

print(f'Batches — Train: {len(train_vloader)} | Val: {len(val_vloader)} | Test: {len(test_vloader)}')

for vids, labels, type_ids in train_vloader:
    print(f'Batch shape: {vids.shape}  Labels: {labels}  Types: {type_ids}')
    break

In [ ]:
# Cell 5 — Load model with image weights (from notebook 03)
# IMPORTANT: Run cells 2-9 of 02_build_model.ipynb first (defines DeepTruthHybridV2)

IMAGE_CKPT = f'{CKPT_DIR}/deeptruth_image_model_final.pth'

model = DeepTruthHybridV2(
    num_fake_types=len(FAKE_TYPE_MAP),
    dropout=0.3
).to(DEVICE)

if os.path.exists(IMAGE_CKPT):
    ckpt = torch.load(IMAGE_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'], strict=False)
    print(f'Loaded image model weights from {IMAGE_CKPT}')
    if 'test_metrics' in ckpt:
        m = ckpt['test_metrics']
        print(f'Image model test AUROC: {m.get("auroc", "N/A"):.4f}')
else:
    print('WARNING: Image checkpoint not found. Run notebook 03 first.')
    print('Starting from scratch.')

criterion = DeepTruthLoss(num_types=len(FAKE_TYPE_MAP), focal_gamma=2.0, label_smoothing=0.1)

total = sum(p.numel() for p in model.parameters())
print(f'Model params: {total/1e6:.1f}M')

In [ ]:
# Cell 6 — Stage 4a: Temporal Transformer warm-up
# Freeze everything except temporal_transformer and video_heads

STAGE4A_EPOCHS = 8
STAGE4A_LR     = 1e-3
STAGE4A_CKPT   = f'{CKPT_DIR}/stage4a_temporal_warmup_best.pth'

# Freeze all except temporal path
for name, param in model.named_parameters():
    if any(k in name for k in ['temporal_transformer', 'video_binary_head', 'video_type_head']):
        param.requires_grad = True
    else:
        param.requires_grad = False

params_4a = [p for p in model.parameters() if p.requires_grad]
print(f'Stage 4a trainable: {sum(p.numel() for p in params_4a)/1e6:.1f}M')

opt4a = torch.optim.AdamW(params_4a, lr=STAGE4A_LR, weight_decay=1e-4)
sched4a = torch.optim.lr_scheduler.OneCycleLR(
    opt4a, max_lr=STAGE4A_LR,
    steps_per_epoch=len(train_vloader),
    epochs=STAGE4A_EPOCHS,
    pct_start=0.2,
)
scaler4a = GradScaler()

best_auroc_4a = 0
history_4a    = []


@torch.no_grad()
def evaluate_video(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_probs, all_labels = [], []
    auroc_metric = AUROC(task='binary').to(device)

    for vids, labels, type_ids in tqdm(loader, desc='Eval', leave=False):
        # vids: (B, T, C, H, W)
        vids, labels, type_ids = vids.to(device), labels.to(device), type_ids.to(device)
        with autocast():
            out = model(vids)  # forward() auto-routes to video path via shape
            loss = criterion(out['binary_logit'], labels, out['type_logit'], type_ids)
        total_loss += loss.item()
        probs = torch.softmax(out['binary_logit'], dim=1)[:, 1]
        all_probs.append(probs)
        all_labels.append(labels)

    all_probs  = torch.cat(all_probs)
    all_labels = torch.cat(all_labels)
    auroc = auroc_metric(all_probs, all_labels).item()
    return {'loss': total_loss / len(loader), 'auroc': auroc}


for epoch in range(STAGE4A_EPOCHS):
    model.train()
    running_loss = 0
    pbar = tqdm(train_vloader, desc=f'Stage4a E{epoch+1}/{STAGE4A_EPOCHS}')

    for vids, labels, type_ids in pbar:
        vids, labels, type_ids = vids.to(DEVICE), labels.to(DEVICE), type_ids.to(DEVICE)
        opt4a.zero_grad()
        with autocast():
            out = model(vids)
            loss = criterion(out['binary_logit'], labels, out['type_logit'], type_ids)
        scaler4a.scale(loss).backward()
        scaler4a.unscale_(opt4a)
        torch.nn.utils.clip_grad_norm_(params_4a, 1.0)
        scaler4a.step(opt4a)
        scaler4a.update()
        sched4a.step()
        running_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    val_m = evaluate_video(model, val_vloader, criterion, DEVICE)
    avg_loss = running_loss / len(train_vloader)
    history_4a.append({'train_loss': avg_loss, **val_m})
    print(f'  4a E{epoch+1}: train={avg_loss:.4f} val_auroc={val_m["auroc"]:.4f}')

    if val_m['auroc'] > best_auroc_4a:
        best_auroc_4a = val_m['auroc']
        torch.save({'model_state': model.state_dict(), 'metrics': val_m}, STAGE4A_CKPT)
        print(f'    -> Best AUROC: {best_auroc_4a:.4f}')

print(f'Stage 4a complete. Best val AUROC: {best_auroc_4a:.4f}')

In [ ]:
# Cell 7 — Stage 4b: Full video fine-tuning
# Unfreeze everything and fine-tune with very low LR

STAGE4B_EPOCHS    = 12
STAGE4B_BASE_LR   = 1e-5
STAGE4B_CKPT      = f'{CKPT_DIR}/stage4b_video_finetune_best.pth'
STAGE4B_LAST_CKPT = f'{CKPT_DIR}/stage4b_video_finetune_last.pth'

# Load best 4a
ckpt = torch.load(STAGE4A_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])

# Unfreeze all
for param in model.parameters():
    param.requires_grad = True

# Layer-wise LR: temporal gets 10x, backbones get 0.1x
backbone_params  = []
temporal_params  = []
fusion_params    = []
for name, param in model.named_parameters():
    if any(k in name for k in ['clip_stream', 'effnet_stream', 'freq_stream', 'srm_stream', 'gram_stream']):
        backbone_params.append(param)
    elif 'temporal' in name:
        temporal_params.append(param)
    else:
        fusion_params.append(param)

opt4b = torch.optim.AdamW([
    {'params': backbone_params, 'lr': STAGE4B_BASE_LR * 0.1},   # 1e-6
    {'params': fusion_params,   'lr': STAGE4B_BASE_LR},          # 1e-5
    {'params': temporal_params, 'lr': STAGE4B_BASE_LR * 5},      # 5e-5
], weight_decay=1e-4)

sched4b = torch.optim.lr_scheduler.CosineAnnealingLR(opt4b, T_max=STAGE4B_EPOCHS, eta_min=1e-8)
scaler4b = GradScaler()

# Early stopping
patience, wait, best_auroc_4b = 5, 0, 0
history_4b = []

print(f'Backbone: {sum(p.numel() for p in backbone_params)/1e6:.1f}M | '
      f'Fusion: {sum(p.numel() for p in fusion_params)/1e6:.1f}M | '
      f'Temporal: {sum(p.numel() for p in temporal_params)/1e6:.1f}M')

for epoch in range(STAGE4B_EPOCHS):
    model.train()
    running_loss = 0
    pbar = tqdm(train_vloader, desc=f'Stage4b E{epoch+1}/{STAGE4B_EPOCHS}')

    for vids, labels, type_ids in pbar:
        vids, labels, type_ids = vids.to(DEVICE), labels.to(DEVICE), type_ids.to(DEVICE)
        opt4b.zero_grad()
        with autocast():
            out = model(vids)
            loss = criterion(out['binary_logit'], labels, out['type_logit'], type_ids)
        scaler4b.scale(loss).backward()
        scaler4b.unscale_(opt4b)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler4b.step(opt4b)
        scaler4b.update()
        running_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    sched4b.step()
    val_m = evaluate_video(model, val_vloader, criterion, DEVICE)
    avg_loss = running_loss / len(train_vloader)
    history_4b.append({'train_loss': avg_loss, **val_m})
    lrs = [f"{g['lr']:.1e}" for g in opt4b.param_groups]
    print(f'  4b E{epoch+1}: train={avg_loss:.4f} val_auroc={val_m["auroc"]:.4f} lrs={lrs}')

    if val_m['auroc'] > best_auroc_4b + 0.001:
        best_auroc_4b = val_m['auroc']
        wait = 0
        torch.save({'model_state': model.state_dict(), 'metrics': val_m, 'epoch': epoch}, STAGE4B_CKPT)
        print(f'    -> Best AUROC: {best_auroc_4b:.4f}')
    else:
        wait += 1
        if wait >= patience:
            print('Early stopping.')
            break

torch.save({'model_state': model.state_dict(), 'metrics': val_m, 'epoch': epoch}, STAGE4B_LAST_CKPT)
print(f'Stage 4b complete. Best val AUROC: {best_auroc_4b:.4f}')

In [ ]:
# Cell 8 — Test set evaluation

ckpt = torch.load(STAGE4B_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded best video checkpoint (epoch {ckpt["epoch"]+1})')

test_m = evaluate_video(model, test_vloader, criterion, DEVICE)
print('\n=== Video Test Set Results ===')
print(f'Loss:  {test_m["loss"]:.4f}')
print(f'AUROC: {test_m["auroc"]:.4f}')

# Frame-level vs video-level accuracy
model.eval()
all_probs, all_labels_list = [], []
with torch.no_grad():
    for vids, labels, _ in tqdm(test_vloader, desc='Video-level eval'):
        vids = vids.to(DEVICE)
        with autocast():
            out = model(vids)
        probs = torch.softmax(out['binary_logit'], dim=1)[:, 1]
        all_probs.extend(probs.cpu().numpy())
        all_labels_list.extend(labels.numpy())

from sklearn.metrics import classification_report, roc_auc_score
preds = [1 if p > 0.5 else 0 for p in all_probs]
print('\nClassification Report:')
print(classification_report(all_labels_list, preds, target_names=['Real', 'Fake']))
print(f'ROC-AUC: {roc_auc_score(all_labels_list, all_probs):.4f}')

In [ ]:
# Cell 9 — Training curves

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

all_losses = [h['train_loss'] for h in history_4a] + [h['train_loss'] for h in history_4b]
all_auroc  = [h['auroc']      for h in history_4a] + [h['auroc']      for h in history_4b]
x = list(range(1, len(all_losses) + 1))

axes[0].plot(x, all_losses, 'b-o', markersize=4)
axes[0].axvline(len(history_4a), color='orange', linestyle='--', label='4a/4b boundary')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Train Loss')
axes[0].set_title('Video Training Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(x, all_auroc, 'g-o', markersize=4)
axes[1].axvline(len(history_4a), color='orange', linestyle='--', label='4a/4b boundary')
axes[1].axhline(test_m['auroc'], color='red', linestyle=':', label=f'Test AUROC={test_m["auroc"]:.3f}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val AUROC')
axes[1].set_title('Video Val AUROC'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('DeepTruth Video Model — Training History', fontsize=14)
plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/training_history_video.png', dpi=150)
plt.show()

In [ ]:
# Cell 10 — Save final video model

FINAL_VIDEO_MODEL = f'{CKPT_DIR}/deeptruth_video_model_final.pth'

torch.save({
    'model_state': model.state_dict(),
    'fake_type_map': FAKE_TYPE_MAP,
    'n_frames': N_FRAMES,
    'img_size': IMG_SIZE,
    'test_metrics': test_m,
    'imagenet_mean': IMAGENET_MEAN,
    'imagenet_std': IMAGENET_STD,
}, FINAL_VIDEO_MODEL)

size_mb = os.path.getsize(FINAL_VIDEO_MODEL) / 1e6
print(f'Final video model saved: {FINAL_VIDEO_MODEL}')
print(f'File size: {size_mb:.1f} MB')
print(f'Test AUROC: {test_m["auroc"]:.4f}')
print('\nReady to proceed to notebook 05 (adversarial hardening + calibration).')